# 00 · Setup and NIM health

**Purpose** — bring the three NIM microservices up on this GPU box and prove they are reachable.

**Prerequisites** — an `nvapi-...` key with NGC catalog access, exported as `NVIDIA_API_KEY`
or placed in `.env` at the repo root. A GPU with >=40GB free (L40S 48GB is the target).

> The key is never printed by this notebook. It is read from the environment and
> only a masked suffix is shown.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Config — key check (masked)

In [ ]:
from getpass import getpass
key = os.environ.get("NVIDIA_API_KEY") or os.environ.get("NGC_API_KEY")
if not key:
    key = getpass("NVIDIA API key (nvapi-...): ")
    os.environ["NVIDIA_API_KEY"] = key
os.environ.setdefault("NGC_API_KEY", key)
assert key.startswith("nvapi-"), "key should start with nvapi-"
print("key loaded:  nvapi-...%s  (len %d)" % (key[-4:], len(key)))

## Config — point the app at the local NIMs

**This is the step that makes the GPU box worth having, and it is easy to skip.**

`.env.example` ships `NIM_MODE=hosted`, and `hosted` does not merely *prefer* the
hosted endpoints — it **skips the local probe entirely**. Leave it and all three
containers can be pulled, built and healthy while every single request still
goes to `integrate.api.nvidia.com`. Nothing errors. The only hint is a note from
`scripts/stack.sh status`: *"3 local NIM container(s) up and NOTHING routes to
them."*

It has to be in `.env`, not just this kernel, because the API and the UI are
separate processes that read `.env` through `scripts/stack.sh`.

Set it **before** notebook 04 builds the index. The embedder changes width with
the mode — `nv-embedqa-e5-v5` is 1024-d, the hosted `nemotron-3-embed-1b` is
2048-d — so building first and switching after means building twice.

In [ ]:
WRITE_ENV = True        # set False to be told what to change instead of changing it
import pathlib, re
envp = pathlib.Path(".env")
cur = os.environ.get("NIM_MODE", "(unset -> auto)")
print("NIM_MODE in this kernel:", cur)
if envp.exists():
    body = envp.read_text()
    found = re.search(r"^NIM_MODE=(.*)$", body, re.M)
    print("NIM_MODE in .env      :", found.group(1) if found else "(absent)")
    if found and found.group(1).strip() == "local":
        print("  already local - nothing to do")
    elif WRITE_ENV:
        body = (re.sub(r"^NIM_MODE=.*$", "NIM_MODE=local", body, count=1, flags=re.M)
                if found else body.rstrip() + "\nNIM_MODE=local\n")
        envp.write_text(body)
        os.environ["NIM_MODE"] = "local"
        print("  -> set NIM_MODE=local in .env")
        print("     restart any already-running stack: bash scripts/stack.sh restart")
    else:
        print("  !! change it by hand:  NIM_MODE=local")
else:
    print("no .env yet - cp .env.example .env, add your key, set NIM_MODE=local")

## Execute — GPU check

If this reports less than ~40GB free, use the 1B reranker instead of the 4B one
(see `scripts/start_nims.sh`, `RRK_IMG`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

## Execute — pull the NIM containers

Long pole: 30–90 minutes including TensorRT engine builds.

**Run it in a terminal, not here.** A `!` cell holds the kernel for its whole
duration, so this one notebook cannot pull *and* let you work through 01 and 02
meanwhile — one kernel runs one cell at a time. In a terminal on the box:

```
cd ~/automotive-service-agent && bash scripts/start_nims.sh pull
```

then come back and carry on. It reads each pull's exit code and fails loudly, so
you do not need to watch it. Left here for when you would rather just block.

In [ ]:
!bash scripts/start_nims.sh pull

## Execute — start the three services

In [ ]:
!bash scripts/start_nims.sh run

## What you should see

All three reporting `OK`, and VRAM around 39–41GB used of 48GB.
First start builds TRT engines, so allow 10–20 minutes before they turn healthy.
Watch one with `!bash scripts/start_nims.sh logs llm`.

In [ ]:
!bash scripts/start_nims.sh health

In [ ]:
# Round-trip test against each endpoint.
import httpx
for name, port, path in [("llm", 8000, "/v1/models"),
                         ("embed", 8001, "/v1/models"),
                         ("rerank", 8002, "/v1/models")]:
    try:
        r = httpx.get(f"http://localhost:{port}{path}", timeout=5)
        ids = [m.get("id") for m in r.json().get("data", [])]
        print(f"  OK      {name:7s} :{port}  -> {ids}")
    except Exception as e:
        print(f"  NOT UP  {name:7s} :{port}  ({type(e).__name__})")

## Execute — the vector store

**No other notebook starts this, and 04 cannot build the index without it.**
`.env` points the app at a standalone Milvus on :19530. If nothing is listening
there the build fails; and if `ASOIA_MILVUS_URI` is unset the app quietly falls
back to an embedded Milvus Lite file in `data/generated/`, which is a *different*
store from the one the API serves — so the index lands somewhere nothing reads.

In [ ]:
!bash scripts/start_milvus.sh up

## Execute — observability and the store admin UIs

Prometheus scrapes the app on :9400 and, because this box has a GPU, the **DCGM
exporter** on :9401 — so the Grafana GPU panels have something to show for the
first time. `provision` is a one-off download of the Attu image and sqlite-web.

Everything binds `127.0.0.1`. Grafana runs anonymous with no login form and
sqlite-web is read-write, so reach them over an `ssh -L` forward and do not
publish these ports.

In [ ]:
!bash scripts/start_observability.sh up

In [ ]:
!bash scripts/stack.sh stores provision && bash scripts/stores.sh up

## What you should see — everything that should be listening

Milvus 19530, Prometheus 9090, Grafana 3000, Attu 8101, sqlite-web 8102, and the
three NIMs on 8000/8001/8002. The API (8080) and UI (7860) come up in 07.

In [ ]:
import socket
for name, port in [("nim-llm", 8000), ("nim-embed", 8001), ("nim-rerank", 8002),
                   ("milvus", 19530), ("prometheus", 9090), ("grafana", 3000),
                   ("dcgm", 9401), ("attu", 8101), ("sqlite-web", 8102)]:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            print(f"  up        {name:12s} :{port}")
    except Exception:
        print(f"  NOT UP    {name:12s} :{port}")